In [ ]:
!pip install -q pymupdf4llm llama-index llama-index-retrievers-bm25

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

import requests
import os
import shutil
import json
import glob
import pprint as pp
import math

from urllib.parse import urlsplit

from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

import pymupdf4llm

from llama_index.core import (
    Settings, SimpleDirectoryReader, VectorStoreIndex, load_index_from_storage
)
from llama_index.core.storage import StorageContext
from llama_index.core.schema import BaseNode
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.core.query_engine import RetrieverQueryEngine, BaseQueryEngine
from llama_index.core.response_synthesizers import get_response_synthesizer
from llama_index.core.base.response.schema import PydanticResponse
from llama_index.core.evaluation import BaseEvaluator, EvaluationResult, BatchEvalRunner
from llama_index.core.llms import LLM, ChatMessage, MessageRole
#from llama_index.core.postprocessor import LongContextReorder

from openai import OpenAI as OpenAIClient

from typing import List, Optional, Literal, Tuple, Final, Dict, Any
#from collections import defaultdict
from pydantic import BaseModel, Field, conint, confloat, ConfigDict

from google.colab import drive

In [ ]:
drive.mount('/content/drive')

os.environ["OPENAI_API_KEY"] = "YOUR KEY HERE"

DATA_DIR: Final = Path('/content/drive/MyDrive/Data/LLM_Climate_Comparison')
CORPORATE_DATA_DIR: Final = DATA_DIR / "corporate_data"
SOURCE_DATA_DIR: Final = CORPORATE_DATA_DIR / "source"
PERSIST_DIR: Final = CORPORATE_DATA_DIR / "persist"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Baseline RAG Approach

OpenAI Hybrid Retrieval w/ Structured Outputs

## Define Structured Output

In [ ]:
# Enums / literals aligned with common SBTi concepts
TargetHorizon = Literal["near_term", "long_term", "net_zero"]
MetricType = Literal["absolute", "intensity"]
Ambition = Literal["1.5C", "well_below_2C", "2C", "unspecified"]
Status = Literal["approved", "committed", "in_validation", "expired", "unknown"]
TargetType = Literal[
    "sbti_near_term",         # reduction target
    "sbti_net_zero",          # long-term/net-zero target
    "non_target_claim"        # neutrality/removals/offsets/SAFc pledges
]

class Target(BaseModel):
  title: Optional[str] = Field(None, description="Human-friendly label if present in text")
  target_type: TargetType = Field(..., description="Classify each item.")
  horizon: TargetHorizon = Field(..., description="near_term / long_term / net_zero (SBTi framing)")
  metric_type: MetricType = Field(..., description="Absolute or intensity target")
  scopes_covered: List[Literal["S1", "S2", "S3"]] = Field(..., description="Which scopes this target covers")
  scope3_categories: Optional[List[conint(ge=1, le=15)]] = Field(
        None, description="If S3: list of GHGP category numbers 1–15, if specified"
  )
  ambition: Ambition = Field(..., description="Declared temperature alignment (if stated)")
  coverage_pct: Optional[confloat(ge=0, le=100)] = Field(
        None, description="Share of emissions covered by the target (company-wide or in-scope)"
  )

  base_year: Optional[int] = Field(None, description="Baseline year (e.g., 2019)")
  target_year: Optional[int] = Field(None, description="Completion year (e.g., 2030)")
  reduction_pct: Optional[confloat(ge=0, le=100)] = Field(
        None, description="Required % reduction vs base (for absolute) or intensity"
  )
  base_value: Optional[float] = Field(
        None, description="If explicitly given (e.g., baseline tCO2e or intensity)"
  )
  target_value: Optional[float] = Field(
        None, description="If explicitly given (e.g., target tCO2e or intensity)"
  )
  unit: Optional[str] = Field(
        None, description="Unit for base/target values (e.g., tCO2e, tCO2e/$, %, ktCO2e)"
  )

  status: Status = Field(..., description="SBTi status if mentioned")
  boundary: Optional[str] = Field(
        None, description="Organizational/operational boundary (e.g., company-wide, specific BU/geography)"
  )

  notes: Optional[str] = Field(None, description="Any qualifiers (market-based/location-based, exclusions, etc.)")
  sources: List[str] = Field(..., description="Doc names/URLs/node IDs where this target was found")

class ExtractedTargets(BaseModel):
  company: Optional[str] = Field(None, description="Name of the company if determinable")
  targets: List[Target] = Field(..., description="All SBTi-style targets found in the context")

class ExtractedTargets(BaseModel):
    company: Optional[str] = None
    targets: List[Target] = Field(
        default_factory=list,
        description="All SBTi-style targets found in the context"
    )


## Establish System Prompt

In [ ]:
SYSTEM_PROMPT = """
You extract structured climate targets data. Output must be valid JSON matching the provided schema—no prose.

Rules:
- Return ALL targets found; do not merge distinct targets.
- Use ONLY the provided context; never infer from general knowledge.
- If a field is not explicitly present, set it to null (or "unknown" where the schema requires).

Target typing & separation:
- horizon ∈ {near_term, long_term, net_zero}
- metric_type ∈ {absolute, intensity}
- ambition ∈ {1.5C, well_below_2C, 2C, unspecified}
- status ∈ {approved, committed, in_validation, expired, unknown}
- Create SEPARATE targets when any of these differ: base_year, target_year, scopes_covered, metric_type, ambition, status, boundary, reduction_pct.

Scopes (strict):
- Do NOT default any scope. Only include a scope if explicitly stated in the same passage as the target.
- Set scopes_covered to [] unless the text clearly states S1/S2/S3 (or “Scope 1/2/3”) for that target.
- If Scope 3 is mentioned, include scope3_categories (1–15) only if explicitly stated; else null.
- Do not backfill scopes from elsewhere.

Coverage & ambition:
- coverage_pct must be numeric only if explicitly stated for that target; else null.
- ambition only if explicitly named (e.g., “1.5°C-aligned”); else "unspecified".

Units & numerics:
- Percent reduction → unit = "%".
- Preserve numeric precision as written.
- If base_value/target_value are present, include unit (e.g., "tCO2e", "tCO2e/$"); else null.

Boundary & notes:
- boundary only if explicitly stated; else null.
- notes: one short sentence of qualifiers as written; no interpretations.

Sources (concrete):
- Populate "sources" with concrete anchors from retrieved nodes, format "file_name#p{page}" (e.g., "Report.pdf#p28"); if no page, use node_id.
- Each target includes ≥1 supporting source anchor.

Empty result:
- If no relevant target data is found, output exactly:
```json
{"company": null, "targets": []}
"""

In [ ]:
SYSTEM_PROMPT = """You extract structured climate target data from the provided context only.

Output:
- Output MUST be valid JSON that matches the provided schema.
- Output JSON only (no prose, no markdown).

Hard constraints:
- Use ONLY the provided context. Never use general knowledge.
- If no qualifying targets are present, output exactly: {"company": null, "targets": []}

Qualifying targets (optimize recall without flooding precision):
Create a target object ONLY if the text explicitly provides either:

A) Quantitative target:
- an explicit time anchor (calendar year like 2030 OR fiscal year like FY25), AND
- at least one explicit quantitative value (reduction_pct OR target_value).

B) Qualitative decarbonization target:
- an explicit time anchor (calendar year or FY), AND
- an explicit keyword: "net zero"/"net-zero"/"carbon neutral"/"climate neutral"/"carbon negative".
For (B), set reduction_pct and target_value to null unless the text explicitly states a numeric value.

If these conditions are not met, DO NOT create a target object.

Local assembly allowed (to handle chunk/table splits safely):
- You may combine information across multiple retrieved snippets ONLY when they share the same concrete source anchor
  (same file_name AND same page) AND it is unambiguous they refer to the same target (same sentence/table row/label).
- Local assembly is allowed ONLY for: target_year (or FY), base_year, reduction_pct, target_value, unit, scopes_covered,
  scope3_categories, coverage_pct, boundary, status, horizon, metric_type, ambition.
- Do NOT combine across different pages or different files/sources.

No guessing:
- If a field is not explicitly stated for that target (after allowed local assembly), set it to null
  (or "unknown" only where the schema requires a non-null string).
- Do NOT infer or backfill from other pages/files.

Splitting / de-duplication:
- Do NOT create duplicates for the same underlying target mentioned multiple times.
- Create SEPARATE targets only when the text explicitly enumerates distinct targets (e.g., different years and/or
  different quantitative values in separate rows/clauses). Do not split based only on null/unknown differences.

Target_type (important for evaluation):
- If a statement qualifies as a target under the rules above, DO NOT label it "non_target_claim".
- Use "non_target_claim" only for statements that do NOT qualify as targets.

Typing rules:
- horizon ∈ {near_term, long_term, net_zero} ONLY if explicitly stated; else null.
- metric_type ∈ {absolute, intensity} ONLY if explicitly stated; else null.
- ambition ∈ {1.5C, well_below_2C, 2C, unspecified}; set only if explicitly named; else "unspecified".
- status ∈ {approved, committed, in_validation, expired, unknown}; set only if explicitly stated; else "unknown".

Scopes (strict):
- scopes_covered must be [] unless the same passage (or allowed same-page assembly) explicitly states S1/S2/S3
  (or “Scope 1/2/3”) for that target.
- If Scope 3 is stated, include scope3_categories (1–15) only if explicitly stated; else null.

Units & numerics:
- Preserve numeric precision as written.
- reduction_pct is ONLY for explicit percent reductions (e.g., "reduce by 42%") and then unit MUST be "%".
- For percent targets that are NOT reductions (e.g., "100% renewable electricity", "67% suppliers engaged"):
  put the percent in target_value and set unit="%" (do NOT use reduction_pct).
- For net zero / carbon neutral / carbon negative targets: target_value MUST be null unless an explicit numeric absolute
  emissions value is stated. Do NOT invent 0.
- If target_value is non-percent and the unit string is explicitly stated in text, use it; else null.

Boundary & notes:
- boundary only if explicitly stated for that target; else null.
- notes: one short sentence capturing qualifiers exactly as written; no interpretation.

Sources (required):
- Every target MUST include >= 1 supporting source anchor from the retrieved nodes.
- Format each anchor as "file_name#p{page}" (e.g., "Report.pdf#p28").
- If page is unavailable, ALWAYS use node_id as the anchor (do not drop the target due to missing page).

Return ALL qualifying targets found in the context. Do not merge distinct, explicitly enumerated targets.

"""

In [ ]:
SYSTEM_PROMPT_LOW_RECALL = """You extract structured climate target data from the provided context only.

Output:
- Output MUST be valid JSON that matches the provided schema.
- Output JSON only (no prose, no markdown).

Hard constraints:
- Use ONLY the provided context. Never use general knowledge.
- If no qualifying targets are present, output exactly: {"company": null, "targets": []}

What counts as a target (to avoid false positives):
- Create a target object ONLY if the passage explicitly provides:
  - a target_year, AND
  - at least one quantitative target number (reduction_pct OR target_value).
- If these are not explicitly present together for a statement, DO NOT create a target for it.

No guessing / no backfilling:
- If a field is not explicitly stated for that target in the same passage, set it to null
  (or "unknown" only where the schema requires a non-null string).
- Do NOT infer or backfill scopes, boundary, status, ambition, units, base_year, horizon, or coverage from elsewhere.

Splitting / de-duplication (critical for precision):
- Do NOT create multiple targets for the same underlying statement.
- Only create SEPARATE target objects when the text explicitly enumerates distinct targets
  (e.g., different target_year and/or different reduction_pct/target_value in separate rows/clauses).
- Differences that arise only because some fields are null/unknown are NOT grounds to split.

Typing rules:
- horizon ∈ {near_term, long_term, net_zero} ONLY if explicitly stated; else null.
- metric_type ∈ {absolute, intensity} ONLY if explicitly stated; else null.
- ambition ∈ {1.5C, well_below_2C, 2C, unspecified}; set only if explicitly named; else "unspecified".
- status ∈ {approved, committed, in_validation, expired, unknown}; set only if explicitly stated; else "unknown".

Scopes (strict):
- scopes_covered must be [] unless the same passage explicitly states S1/S2/S3 (or “Scope 1/2/3”) for that target.
- If Scope 3 is stated, include scope3_categories (1–15) only if explicitly stated; else null.

Units & numerics:
- Percent reduction => unit = "%".
- Preserve numeric precision as written.
- If base_value or target_value is present, include the explicit unit string from the text; else null.
- Do NOT invent units.

Boundary & notes:
- boundary only if explicitly stated for that target; else null.
- notes: one short sentence capturing qualifiers exactly as written; no interpretation.

Sources (required):
- Every target MUST include >= 1 supporting source anchor from the retrieved nodes.
- Format each source anchor as "file_name#p{page}" (e.g., "Report.pdf#p28"); if no page, use node_id.
- If you cannot provide a concrete source anchor for a target, DO NOT output that target.

Return ALL qualifying targets found in the context. Do not merge distinct, explicitly enumerated targets.

"""

In [ ]:
SYSTEM_PROMPT_OVER_EXTRACT = """You extract structured climate target data from the provided context only.

Output:
- Output MUST be valid JSON that matches the provided schema.
- Output JSON only (no prose, no markdown).

Hard constraints:
- Use ONLY the provided context. Never use general knowledge.
- If no qualifying targets are present, output exactly: {"company": null, "targets": []}

No guessing / no backfilling:
- If a field is not explicitly stated for that target in the same passage, set it to null
  (or "unknown" only where the schema requires a non-null string).

Splitting / de-duplication (critical for precision):
- Do NOT create multiple targets for the same underlying statement.
- Only create SEPARATE target objects when the text explicitly enumerates distinct targets
  (e.g., different target_year and/or different reduction_pct/target_value for different scopes in separate rows/clauses).
- Differences that arise only because some fields are null/unknown are NOT grounds to split.

Typing rules:
- horizon ∈ {near_term, long_term, net_zero} ONLY if explicitly stated; else null.
- metric_type ∈ {absolute, intensity} ONLY if explicitly stated; else null.
- ambition ∈ {1.5C, well_below_2C, 2C, unspecified}; set only if explicitly named; else "unspecified".
- status ∈ {approved, committed, in_validation, expired, unknown}; set only if explicitly stated; else "unknown".

Units & numerics:
- Percent reduction => unit = "%".
- Preserve numeric precision as written.
- If base_value or target_value is present, include the explicit unit string from the text; else null.
- Do NOT invent units.

Boundary & notes:
- boundary only if explicitly stated for that target; else null.
- notes: one short sentence capturing qualifiers exactly as written; no interpretation.

Sources (required):
- Every target MUST include >= 1 supporting source anchor from the retrieved nodes.
- Format each source anchor as "file_name#p{page}" (e.g., "Report.pdf#p28"); if no page, use node_id.

Return ALL qualifying targets found in the context. Do not merge distinct, explicitly enumerated targets.

"""

In [ ]:
SYSTEM_PROMPT="""You extract structured climate target data from the provided context only.

Output:
- Output MUST be valid JSON that matches the provided schema.
- Output JSON only (no prose, no markdown).

Hard constraints:
- Use ONLY the provided context; never use general knowledge.
- If no qualifying targets are present, output exactly: {"company": null, "targets": []}

What counts as a target (optimize recall with minimal precision loss):
Create a target object ONLY if EITHER:

1) Quantitative target:
- the retrieved context explicitly provides a time anchor (calendar year like 2030 OR fiscal year like FY25), AND
- the retrieved context explicitly provides a quantitative value (reduction_pct OR target_value),
AND it is unambiguous they refer to the same target.

You may combine the time anchor and quantitative value across snippets ONLY if they share the same source anchor
(same file_name AND same page) and clearly refer to the same sentence/table row/label.
Do NOT combine across different pages or different files.

2) Qualitative net-zero-style target:
- the text explicitly contains one of: "net zero", "net-zero", "carbon neutral", "climate neutral", "carbon negative", AND
- an explicit time anchor (calendar year or FY).
For these, set reduction_pct and target_value to null unless an explicit numeric value is stated.

If these conditions are not met, DO NOT create a target object.

No guessing / no backfilling:
- If a field is not explicitly stated for that target (after the allowed same-page combination above), set it to null
  (or "unknown" only where the schema requires a non-null string).
- Do NOT infer or backfill scopes, boundary, status, ambition, units, base_year, horizon, or coverage from other pages/files.

Splitting / de-duplication (critical for precision):
- Do NOT create multiple targets for the same underlying statement.
- Only create SEPARATE target objects when the text explicitly enumerates distinct targets
  (e.g., different target years and/or different quantitative values in separate rows/clauses).
- Differences that arise only because some fields are null/unknown are NOT grounds to split.

Typing rules:
- horizon ∈ {near_term, long_term, net_zero} ONLY if explicitly stated; else null.
- metric_type ∈ {absolute, intensity} ONLY if explicitly stated; else null.
- ambition ∈ {1.5C, well_below_2C, 2C, unspecified}; set only if explicitly named; else "unspecified".
- status ∈ {approved, committed, in_validation, expired, unknown}; set only if explicitly stated; else "unknown".

Scopes (strict):
- scopes_covered must be [] unless the same sentence/table row (or allowed same-page combination) explicitly states
  S1/S2/S3 (or “Scope 1/2/3”) for that target.
- If Scope 3 is stated, include scope3_categories (1–15) only if explicitly stated; else null.

Units & numerics:
- Percent reduction => reduction_pct with unit = "%".
- Preserve numeric precision as written.
- If target_value is present, include the explicit unit string from the text; else null.
- Do NOT invent units or numeric values.

Boundary & notes:
- boundary only if explicitly stated; else null.
- notes: one short sentence capturing qualifiers exactly as written; no interpretation.

Sources (required):
- Every target MUST include >= 1 supporting source anchor from the retrieved nodes.
- Use "file_name#p{page}" when page is available; otherwise ALWAYS use node_id.
- Do NOT drop a target solely because a page number is missing.

Return ALL qualifying targets found in the context. Do not merge distinct, explicitly enumerated targets.

"""

## Build RAG pipeline

In [ ]:
QUERY_PROMPT : Final = (
  "Extract climate emission target data that follows SBTi-style definitions "
  "(near-term, long-term/net-zero; absolute or intensity; Scopes 1/2/3), "
  "These could be in text form, covering any of the areas of Scope 1, 2 or 3"
  "from electricity to value chain emissions."
  "STRICTLY from the provided context. If a field isn’t stated, leave it null."
)

In [ ]:
def build_or_load_vector_index(
  input_dir: Path,
  persist_dir: Path,
  *,
  rebuild: bool = False,
  embed_model: str = "text-embedding-3-large",
  chunk_size: int = 8192,
  chunk_overlap: int = 512,
) -> Tuple[VectorStoreIndex, List[BaseNode]]:
  """Create vector index if it doesn't exist, otherwise load it.
  Returns (index, nodes)."""

  Settings.embed_model = OpenAIEmbedding(
                                        model=embed_model,
                                        timeout=60,
                                        max_retries=5)
  Settings.text_splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

  persist_dir.mkdir(parents=True, exist_ok=True)
  storage_ready = any(persist_dir.iterdir()) and not rebuild

  if storage_ready:
    storage_ctx = StorageContext.from_defaults(persist_dir=str(persist_dir))
    vector_index = load_index_from_storage(storage_ctx)
    docstore = vector_index.storage_context.docstore
    nodes = list(docstore.docs.values())
  else:
    pdf_reader = pymupdf4llm.LlamaMarkdownReader()
    extractors = {".pdf": pdf_reader}  # .txt/.md handled by SimpleDirectoryReader
    docs = SimpleDirectoryReader(input_dir=str(input_dir),
                                 file_extractor=extractors).load_data()
                                 # set metadata like ticker, year if needed
    pipeline = IngestionPipeline(
      transformations=[
        Settings.text_splitter,
        # add further custom transformations here - e.g. KeywordExtractors, etc
      ]
    )
    nodes = pipeline.run(documents=docs)
    vector_index = VectorStoreIndex(nodes)
    vector_index.storage_context.persist(persist_dir=str(persist_dir))
  return vector_index, nodes

def make_structured_query_engine(
  vector_index: VectorStoreIndex,
  nodes: List[BaseNode],
  *,
  chat_model: str,
  temperature: float = 0.0,
  max_tokens: int = 16390,
  similarity_top_k: int = 10,
  fusion_top_k: int = 5
) -> RetrieverQueryEngine:
  """Create a hybrid (vector + BM25) query engine with reciprocal-rank fusion
  with structured outputs"""

  base_llm = OpenAI(
    model=chat_model,
    temperature=temperature,
    max_tokens=max_tokens,
    timeout=300,
    strict=True,
    max_retries=5,
    system_prompt=SYSTEM_PROMPT,
  )

  structured_llm = base_llm.as_structured_llm(ExtractedTargets)
  synth = get_response_synthesizer(llm=structured_llm,
                                   response_mode="compact"
                                   )

  vector_retriever = vector_index.as_retriever(similarity_top_k=similarity_top_k)
  bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=similarity_top_k)

  hybrid = QueryFusionRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    num_queries=1, # avoid query rewrites
    similarity_top_k=fusion_top_k,
    mode="reciprocal_rerank",
    use_async=False,
  )

  qe = RetrieverQueryEngine(retriever=hybrid,
                            #node_postprocessors=postprocs,
                            response_synthesizer=synth)

  return qe

def get_qe_for_company_year(
  company_ticker: str,
  year: str,
  model: str,
  rebuild: bool = False
  ) -> RetrieverQueryEngine:
  """ returns a query engine for a specific company and year
      folder of documents.
  """
  company_source_dir = SOURCE_DATA_DIR / company_ticker / year
  index_persist_dir = PERSIST_DIR / "indexes" / company_ticker / year

  vector_index, nodes = build_or_load_vector_index(company_source_dir, index_persist_dir, rebuild=rebuild)
  structured_hybrid_qe = make_structured_query_engine(vector_index=vector_index,
                                                    chat_model=model,
                                                    nodes=nodes)
  return structured_hybrid_qe

In [ ]:
def write_company_year_json(
  company_ticker: str,
  year: str,
  model_obj: ExtractedTargets
) -> Path:
  """ writes the json output for the QUERY_PROMPT
      response for a given company and year
  """

  output_persist_dir = PERSIST_DIR / "output"
  output_persist_dir.mkdir(parents=True, exist_ok=True)

  schema_version = "v1"
  out_path = output_persist_dir / f"{company_ticker}.{year}.targets.{schema_version}.json"
  payload = model_obj.model_dump(mode="json")

  json_str = json.dumps(
    payload,
    ensure_ascii=False,
    indent=2,
    sort_keys=True,
    allow_nan=False,)

  out_path.write_text(json_str + "\n", encoding="utf-8")
  print(f"wrote {out_path}")

  return out_path

In [ ]:
company_ticker = "aapl"
year = "2023"

def run_company_year(company_ticker, year, model, rebuild=False):
  company_ticker = company_ticker.lower()
  structured_hybrid_qe = get_qe_for_company_year(company_ticker=company_ticker, year=year, model=model, rebuild=False)
  res = structured_hybrid_qe.query(QUERY_PROMPT)
  model_obj = res.response
  nodes = res.source_nodes
  pp.pprint(model_obj.model_dump())
  return model_obj, nodes

nodes = False
#model_obj, nodes = run_company_year(company_ticker, year, model = "gpt-5-2025-08-07")
if nodes:
  for node in nodes:
    print(node.metadata["page"])
#_ = write_company_year_json(company_ticker, year, model_obj)


# Evaluation - LLM as Judge

In [ ]:
FIELDS_TO_SCORE = [
  "ambition", "base_year", "horizon",
  "metric_type","reduction_pct",
  "scopes_covered", "target_year",
  "target_value", "unit"
]

EVAL_SYSTEM_PROMPT = """
You are an expert evaluator of structured climate targets (SBTi-style).
Your job: (1) ALIGN predicted targets to gold targets (1:1 matches), ignoring any target whose target_type = "non_target_claim";
(2) SCORE ONLY these fields: ambition, base_year, horizon, metric_type, reduction_pct, scopes_covered, target_year, target_value, unit
(3) Be robust to formatting differences, synonyms, and minor numeric/rounding differences.
(4) IMPORTANT: Perform both MATCHING and SCORING **using only the listed fields**; treat all other fields as unavailable.
(5) Use the rubric:
- EXACT = clearly the same value/meaning (case/format differences OK; synonyms OK; minor numeric diffs OK: ±0.5 abs OR ±1% relative)
- PARTIAL = meaning overlaps but is not fully correct (e.g., scopes partially overlap, or ambition “well_below_2C” vs “2C”)
- WRONG = materially different or missing when present in gold

Match policy:
- Create pairs between gold and predicted targets that represent the same real-world target.
- Prefer pairs that share the same target_type, horizon, and scope coverage.
- Each gold can match at most one prediction; each prediction can match at most one gold.

Return STRICT JSON:
{
  "matches": [
    {
      "gold_index": <int>,
      "pred_index": <int>,
      "field_scores": {
        "<field>": {"grade": "EXACT|PARTIAL|WRONG", "note": "<short reason>"}, ...
      }
    },
    ...
  ],
  "unmatched_gold": [<int>, ...],
  "unmatched_pred": [<int>, ...]
}
No extra keys. No commentary outside JSON.
"""

USER_TEMPLATE = """
Gold JSON for one document:
{gold}

Predicted JSON for the same document:
{pred}

Remember:
- Ignore targets where target_type == "non_target_claim" on BOTH sides.
- Only score these fields: {fields}.
- Return STRICT JSON as specified.
"""

In [ ]:
Grade = Literal["EXACT", "PARTIAL", "WRONG"]

class FieldScore(BaseModel):
  model_config = ConfigDict(extra="forbid")
  grade: Grade
  note: str

class FieldScores(BaseModel):
  # This forces exactly these keys (no more, no less)
  model_config = ConfigDict(extra="forbid")
  ambition: FieldScore
  base_year: FieldScore
  horizon: FieldScore
  metric_type: FieldScore
  reduction_pct: FieldScore
  scopes_covered: FieldScore
  target_year: FieldScore
  target_value: FieldScore
  unit: FieldScore

class Match(BaseModel):
  model_config = ConfigDict(extra="forbid")
  gold_index: int
  pred_index: int
  field_scores: FieldScores

class ScoreOutput(BaseModel):
  model_config = ConfigDict(extra="forbid")
  matches: List[Match]
  unmatched_gold: List[int]
  unmatched_pred: List[int]


def judge_align_and_score_openai(gold: dict, pred: dict, model="gpt-5-mini-2025-08-07") -> dict:
  client = OpenAIClient(api_key=os.getenv("OPENAI_API_KEY"))

  prompt = USER_TEMPLATE.format(
        gold=json.dumps(_project(gold), ensure_ascii=False),
        pred=json.dumps(_project(pred), ensure_ascii=False),
        fields=", ".join(FIELDS_TO_SCORE),
    )

  resp = client.responses.parse(
        model=model,
        input=[
            {"role": "system", "content": EVAL_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        text_format=ScoreOutput,
        reasoning={"effort": "high"},
    )

  parsed = getattr(resp, "output_parsed", None)

  if parsed is not None:
    return parsed.model_dump()

  txt = (getattr(resp, "output_text", None) or "").strip()
  try:
    return json.loads(txt)
  except:  # salvage JSON if model adds stray text
    i,j = txt.find("{"), txt.rfind("}")
    return json.loads(txt[i:j+1])

def _project(obj: dict) -> dict:
  """
    keep only certain fields for scoring
  """
  keep = set(FIELDS_TO_SCORE + ["target_type"])
  return {
    "company": obj.get("company"),
    "targets": [
      {k: t.get(k) for k in keep if k in t}
      for t in obj.get("targets", []) if t.get("target_type") != "non_target_claim"
    ],
  }

def _grade_to_score(g) -> float:
  """ convert a grade to a score
  """
  return 1.0 if g=="EXACT" else (0.5 if g=="PARTIAL" else 0.0)

def evaluate_dataset(test_cases: List[Dict[str, Any]]) -> Dict[str, Any]:
  """ evaluate a list of test cases.
  test_cases: [{"doc_name": str, "response": pred_json_str, "reference": gold_json_str}, ...]
  """
  per_doc = []
  tp = fp = fn = 0
  sums = {f:0.0 for f in FIELDS_TO_SCORE}
  cnts = {f:0 for f in FIELDS_TO_SCORE}

  for test_case in test_cases:
    doc = test_case["doc_name"]
    gold = json.loads(test_case["reference"])
    pred = json.loads(test_case["response"])
    judged = judge_align_and_score_openai(gold, pred)

    matches = judged.get("matches", [])
    ug = judged.get("unmatched_gold", [])
    up = judged.get("unmatched_pred", [])
    tp += len(matches)
    fp += len(up)
    fn += len(ug)

    field_acc = {f: None for f in FIELDS_TO_SCORE}
    for f in FIELDS_TO_SCORE:
      ok = [
        _grade_to_score(m.get("field_scores", {}).get(f, {}).get("grade", "WRONG"))
        for m in matches if f in m.get("field_scores", {})
      ]
      if ok:
        v = sum(ok)/len(ok)
        field_acc[f]=v
        sums[f]+=v
        cnts[f]+=1

    per_doc.append({"doc": doc, "tp": len(matches), "fp": len(up), "fn": len(ug), "field_accuracy": field_acc})

  micro_p = tp/(tp+fp) if tp+fp else 1.0
  micro_r = tp/(tp+fn) if tp+fn else 1.0
  micro_f1 = 2*micro_p*micro_r/(micro_p+micro_r) if micro_p+micro_r else 0.0
  hallucination_rate = fp/(tp+fp) if tp+fp else 0.0

  field_macro = {f:(sums[f]/cnts[f] if cnts[f] else None) for f in FIELDS_TO_SCORE}

  return {"aggregate":{
              "micro_precision": micro_p,
              "micro_recall": micro_r,
              "micro_f1": micro_f1,
              "hallucination_rate": hallucination_rate,  # <-- and return it
              "field_acc_macro": field_macro
          },
          "per_doc": per_doc}

In [ ]:
def get_json_pairs(
    company_ticker: str,
    year: str,
  ) -> Tuple[str, str, str]:
  output_dir = PERSIST_DIR / "output"
  validation_dir = PERSIST_DIR / "validation_set"

  doc_name = f"{company_ticker}.{year}.targets.v1.json"

  test_json_path = output_dir / doc_name
  validation_json_path = validation_dir / doc_name

  test_json_str = json.dumps(json.loads(test_json_path.read_text()), ensure_ascii=False)
  validation_json_str = json.dumps(json.loads(validation_json_path.read_text()), ensure_ascii=False)

  return doc_name, test_json_str, validation_json_str

In [ ]:
def get_test_cases(
    company_tickers : List[str],
    years: List[str]
  ) -> List[dict]:
  test_cases = []
  for company_ticker in company_tickers:
    for year in years:
      company_ticker = company_ticker.lower()
      print(company_ticker, year)
      doc_name, test_json_str, validation_json_str = get_json_pairs(company_ticker, year)
      test_cases.append({
        "doc_name": doc_name,
        "response": test_json_str,
        "reference": validation_json_str,
      })
  return test_cases

In [ ]:
def write_evaluation_json(
  filename: str,
  evaluation_dict: dict
) -> Path:
  """ writes the json output for the evaluation
  """

  output_persist_dir = PERSIST_DIR / "evaluation"
  output_persist_dir.mkdir(parents=True, exist_ok=True)

  out_path = output_persist_dir / f"{filename}.json"

  json_str = json.dumps(
    evaluation_dict,
    ensure_ascii=False,
    indent=2,
    sort_keys=True,
    allow_nan=False,)

  out_path.write_text(json_str + "\n", encoding="utf-8")
  print(f"wrote {out_path}")

  return out_path

# Batch Run

In [ ]:
def copy_dir_contents(src, dst):
  src, dst = Path(src), Path(dst)
  dst.mkdir(parents=True, exist_ok=True)

  for p in src.iterdir():
    target = dst / p.name
    if p.is_dir():
      shutil.copytree(p, target, dirs_exist_ok=True)
    else:
      shutil.copy2(p, target)

In [ ]:
def batch_run(
  years: List[str],
  company_tickers: List[str],
  model: str,
  rebuild: bool = False
  ):
  for company_ticker in company_tickers:
    for year in years:
      company_ticker = company_ticker.lower()
      print("processing:", company_ticker, year, model)
      model_obj, nodes = run_company_year(company_ticker, year, model)
      _ = write_company_year_json(company_ticker, year, model_obj)

rebuild=False

models = {
     "gpt5" : "gpt-5-2025-08-07",
     "gpt5_1" : "gpt-5.1-2025-11-13",
     "gpt5_2" : "gpt-5.2-2025-12-11",
}

# set model here
model_name = "gpt5_2"
model = models[model_name]

company_info = {
    "NVDA": "NVIDIA Corporation",
    "MSFT": "Microsoft Corporation",
    "AAPL": "Apple Inc",
    "GOOGL": "Alphabet Inc",
    "AMZN": "Amazon.com Inc",
    "META": "Meta Platforms Inc",
    "TSLA": "Tesla Inc",
}

years = ["2023", "2024"]
company_tickers = list(company_info.keys())

repeats = 3
reports = []

for i in range(repeats):
  output_dir = PERSIST_DIR / "output"
  output_dir.mkdir(parents=True, exist_ok=True)
  shutil.rmtree(output_dir, ignore_errors=True)
  counter = i+1
  print("running rag", counter, "for", model)
  batch_run(years, company_tickers, model, rebuild)
  dest_dir = PERSIST_DIR / "rag_output" / model_name / f"run_{counter}"
  copy_dir_contents(output_dir, dest_dir )
  print("running eval", counter)
  test_cases = get_test_cases(company_tickers, years)
  report = evaluate_dataset(test_cases)
  reports.append(report)
  write_evaluation_json(f"v2_{model_name}_{counter}",report)

running rag 1 for gpt-5.2-2025-12-11
processing: nvda 2023 gpt-5.2-2025-12-11


DEBUG:bm25s:Building index from IDs objects


{'company': 'NVIDIA',
 'targets': [{'ambition': '1.5C',
              'base_value': None,
              'base_year': None,
              'boundary': 'our operations and data centers',
              'coverage_pct': None,
              'horizon': 'near_term',
              'metric_type': 'absolute',
              'notes': 'By FY25, and annually thereafter, achieve and maintain '
                       '100% renewable electricity for our operations and data '
                       'centers; commit to procure renewable energy from '
                       'sources that create additionality when available and '
                       'commercially reasonable.',
              'reduction_pct': None,
              'scope3_categories': None,
              'scopes_covered': ['S1', 'S2'],
              'sources': ['FY2023-NVIDIA-Corporate-Responsibility-Report-1.pdf#p37'],
              'status': 'committed',
              'target_type': 'non_target_claim',
              'target_value': 100.0,
 

DEBUG:bm25s:Building index from IDs objects


{'company': None, 'targets': []}
wrote /content/drive/MyDrive/Data/LLM_Climate_Comparison/corporate_data/persist/output/nvda.2024.targets.v1.json
processing: msft 2023 gpt-5.2-2025-12-11


...

{'company': 'Meta',
 'targets': [{'ambition': 'unspecified',
              'base_value': None,
              'base_year': 2021,
              'boundary': None,
              'coverage_pct': None,
              'horizon': 'near_term',
              'metric_type': 'absolute',
              'notes': None,
              'reduction_pct': 42.0,
              'scope3_categories': None,
              'scopes_covered': ['S1', 'S2'],
              'sources': ['Meta_2025-Sustainability-Report.pdf#p8'],
              'status': 'unknown',
              'target_type': 'sbti_near_term',
              'target_value': None,
              'target_year': 2031,
              'title': 'Reduce our Scope 1 and 2 emissions by 42% in 2031 from '
                       'a 2021 baseline',
              'unit': '%'},
             {'ambition': 'unspecified',
              'base_value': None,
              'base_year': None,
              'boundary': None,
              'coverage_pct': None,
              'horizon'

DEBUG:bm25s:Building index from IDs objects


{'company': None, 'targets': []}
wrote /content/drive/MyDrive/Data/LLM_Climate_Comparison/corporate_data/persist/output/tsla.2023.targets.v1.json
processing: tsla 2024 gpt-5.2-2025-12-11


DEBUG:bm25s:Building index from IDs objects


{'company': None,
 'targets': [{'ambition': 'unspecified',
              'base_value': None,
              'base_year': None,
              'boundary': None,
              'coverage_pct': None,
              'horizon': 'net_zero',
              'metric_type': 'absolute',
              'notes': 'We strive to achieve net-zero GHG emissions.',
              'reduction_pct': None,
              'scope3_categories': None,
              'scopes_covered': [],
              'sources': ['2024-tesla-impact-report-highlights.pdf#p7'],
              'status': 'unknown',
              'target_type': 'sbti_net_zero',
              'target_value': None,
              'target_year': None,
              'title': 'net-zero GHG emissions',
              'unit': None}]}
wrote /content/drive/MyDrive/Data/LLM_Climate_Comparison/corporate_data/persist/output/tsla.2024.targets.v1.json
running eval 3
nvda 2023
nvda 2024
msft 2023
msft 2024
aapl 2023
aapl 2024
googl 2023
googl 2024
amzn 2023
amzn 2024
meta 2023

In [ ]:
for report in reports:
  print(json.dumps(report["aggregate"], indent=2))

{
  "micro_precision": 0.9473684210526315,
  "micro_recall": 0.72,
  "micro_f1": 0.8181818181818181,
  "hallucination_rate": 0.05263157894736842,
  "field_acc_macro": {
    "ambition": 0.7222222222222222,
    "base_year": 0.925925925925926,
    "horizon": 0.8703703703703703,
    "metric_type": 1.0,
    "reduction_pct": 0.9259259259259258,
    "scopes_covered": 0.6666666666666666,
    "target_year": 0.9629629629629629,
    "target_value": 0.9259259259259258,
    "unit": 0.537037037037037
  }
}
{
  "micro_precision": 0.7692307692307693,
  "micro_recall": 0.8333333333333334,
  "micro_f1": 0.8,
  "hallucination_rate": 0.23076923076923078,
  "field_acc_macro": {
    "ambition": 0.75,
    "base_year": 0.9333333333333333,
    "horizon": 0.9166666666666666,
    "metric_type": 1.0,
    "reduction_pct": 0.8833333333333332,
    "scopes_covered": 0.6583333333333333,
    "target_year": 0.9166666666666666,
    "target_value": 0.8833333333333332,
    "unit": 0.5333333333333333
  }
}
{
  "micro_precis

In [ ]:
def mean_std_micro(results, ddof=1):
  keys = ("micro_precision", "micro_recall", "micro_f1", "hallucination_rate")
  out = {}
  for k in keys:
    xs = np.array([r["aggregate"][k] for r in results], dtype=float)
    out[k] = {
            "mean": float(xs.mean()),
            "std": float(xs.std(ddof=ddof)),
            "n": int(xs.size),
      }
  return out

# mean

In [ ]:
mean_std_micro(reports, ddof=1)

{'micro_precision': {'mean': 0.8467095340160355,
  'std': 0.0913030355456033,
  'n': 3},
 'micro_recall': {'mean': 0.7298989898989898,
  'std': 0.09885726013727467,
  'n': 3},
 'micro_f1': {'mean': 0.7787101787101788, 'std': 0.05340047718418885, 'n': 3},
 'hallucination_rate': {'mean': 0.15329046598396445,
  'std': 0.09130303554560334,
  'n': 3}}

In [ ]:
report

{'aggregate': {'micro_precision': 0.8235294117647058,
  'micro_recall': 0.6363636363636364,
  'micro_f1': 0.717948717948718,
  'hallucination_rate': 0.17647058823529413,
  'field_acc_macro': {'ambition': 0.8125,
   'base_year': 0.9583333333333333,
   'horizon': 0.9583333333333333,
   'metric_type': 1.0,
   'reduction_pct': 0.9583333333333333,
   'scopes_covered': 0.625,
   'target_year': 1.0,
   'target_value': 0.7708333333333333,
   'unit': 0.4583333333333333}},
 'per_doc': [{'doc': 'nvda.2023.targets.v1.json',
   'tp': 1,
   'fp': 1,
   'fn': 1,
   'field_accuracy': {'ambition': 1.0,
    'base_year': 1.0,
    'horizon': 1.0,
    'metric_type': 1.0,
    'reduction_pct': 1.0,
    'scopes_covered': 1.0,
    'target_year': 1.0,
    'target_value': 0.0,
    'unit': 0.0}},
  {'doc': 'nvda.2024.targets.v1.json',
   'tp': 0,
   'fp': 0,
   'fn': 2,
   'field_accuracy': {'ambition': None,
    'base_year': None,
    'horizon': None,
    'metric_type': None,
    'reduction_pct': None,
    'scop